# Detecção de Imagens Sintéticas após Transformações

**Objetivo:** medir a robustez de features clássicas (FFT, HOG, LBP) + SVM no CIFAKE quando as imagens sofrem compressão, crop, resize, blur etc.

**Fluxo:** carregar dados → extrair features (só originais) → treinar SVM → aplicar transformações → medir queda de acurácia por feature.

> Desenvolvedores: Igor de Matos da Rosa e Nathália Geraldino Ribas

## 1. Setup e caminhos
Preparar o ambiente e apontar para o CIFAKE.

In [ ]:
# Este bloco visa importar bibliotecas necessárias:
# - random (aleatoriedade)
# - numpy, pandas (dados e tabelas)
# - PIL / opencv (ler e transformar imagens)
# - skimage (HOG, LBP)
# - sklearn (SVM, métricas, StandardScaler)
# - matplotlib / seaborn (gráficos)
# - pathlib / os (caminhos)
# - joblib (salvar modelos .pkl)   
import os
import random
from pathlib import Path

import cv2
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from PIL import Image
from skimage.feature import hog, local_binary_pattern
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

In [1]:
# Define caminhos do projeto:
# - pasta raiz do repositório
# - caminho do CIFAKE ../Base/archive (bifurca em teste e treino)
# - pastas de saída: models/ e results/
# Fixar seed aleatória para reprodutibilidade

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Caminhos do projeto
ROOT_DIR = Path.cwd()

DATASET_DIR = ROOT_DIR.parent / "Base" / "archive"
TRAIN_DIR = DATASET_DIR / "train"
TEST_DIR = DATASET_DIR / "test"

# Pastas de saída: models/ e results/
MODELS_DIR = ROOT_DIR / "models"
RESULTS_DIR = ROOT_DIR / "results"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

NameError: name 'random' is not defined

## 2. Carregar o CIFAKE
Dataset balanceado REAL/FAKE, já separado em treino e teste.

In [ ]:
# Função para listar imagens das pastas REAL/ e FAKE/
# Retornar: lista de caminhos + rótulos (0=FAKE, 1=REAL ou o padrão que escolherem)
# Aplicar para train/ e test/
# Conferir quantas imagens há em cada classe (deve estar balanceado)
# (Opcional no início) usar um SUBCONJUNTO menor para prototipar rápido

In [ ]:
# Função para carregar uma imagem:
# - abrir arquivo (PIL ou cv2)
# - garantir tamanho consistente (CIFAKE já é 32x32; documentar se for redimensionar)
# - converter para o formato usado nas features (RGB ou cinza, conforme o caso)
# Mostrar 4–8 exemplos (REAL e FAKE) só para inspeção visual

## 3. Extratores de features
Três famílias alinhadas à disciplina: **frequência (FFT)**, **espaço (HOG)**, **textura (LBP)**.

In [ ]:
# Implementar extrator FFT:
# - converter imagem para cinza (se necessário)
# - calcular FFT 2D e espectro de magnitude
# - (opcional) shift para centralizar baixas frequências
# - transformar em vetor 1D (ex.: histograma do espectro, ou achatamento + redução)
# - normalizar o vetor
# Testar em 1 imagem e imprimir o shape do vetor

In [ ]:
# Implementar extrator HOG:
# - usar skimage.feature.hog (ou OpenCV)
# - definir parâmetros razoáveis para 32x32 (pixels_per_cell, cells_per_block, orientations)
# - retornar o vetor de features
# Testar em 1 imagem e imprimir o shape do vetor

In [ ]:
# Implementar extrator LBP:
# - usar skimage.feature.local_binary_pattern
# - escolher P (vizinhos) e R (raio)
# - montar histograma do LBP como vetor de features
# - normalizar o histograma
# Testar em 1 imagem e imprimir o shape do vetor

In [ ]:
# Função genérica: dado uma lista de caminhos + nome da feature ("fft"|"hog"|"lbp"),
# extrair features de todas as imagens e empilhar em matriz X
# Retornar X (n_amostras x n_features) e y (rótulos)
# Mostrar progresso (tqdm) — CIFAKE é grande
# (Opcional) salvar matrizes em disco (npz/npz) para não recalcular sempre

## 4. Features do conjunto ORIGINAL (treino e teste)
**Importante:** o SVM treina só em imagens **sem** transformação (baseline).

In [ ]:
# Para cada tipo de feature (FFT, HOG, LBP):
# - extrair X_train, y_train a partir das imagens ORIGINAIS de treino
# - extrair X_test, y_test a partir das imagens ORIGINAIS de teste
# - aplicar StandardScaler (fit só no treino; transform no teste)
# Guardar em dicionários, ex.: features["hog"]["X_train"], etc.

## 5. Treinar SVM (baseline)
Um SVM por família de feature, treinado só nos originais.

In [ ]:
# Para cada feature (FFT, HOG, LBP):
# - criar SVM (começar com kernel linear ou RBF; C padrão ou grid pequeno)
# - treinar com X_train / y_train ORIGINAIS
# - prever no X_test ORIGINAL
# - calcular acurácia (e opcionalmente precision/recall/f1)
# Guardar modelo e métricas baseline em um dicionário
# Imprimir tabela: feature | acurácia baseline

In [ ]:
# Salvar os 3 modelos SVM em models/ (joblib.dump)
# Ex.: models/svm_fft.pkl, models/svm_hog.pkl, models/svm_lbp.pkl
# Salvar também os scalers, se usados
# Documentar no README depois o caminho desses arquivos (entrega do Moodle pede isso)

## 6. Transformações digitais
Simular o que acontece quando a imagem circula na internet.

In [ ]:
# Implementar função jpeg_compress(imagem, qualidade):
# - salvar/reabrir em JPEG com qualidade 90, 70 e 50 (ou simular em memória)
# - retornar imagem comprimida no mesmo formato das features
# Testar visualmente em 1 exemplo REAL e 1 FAKE

In [ ]:
# Implementar função crop_center(imagem, percentual):
# - cortar 10%, 25% e 40% (mantendo centro ou crop aleatório — documentar a escolha)
# - redimensionar de volta para o tamanho original se as features exigirem tamanho fixo
# Testar visualmente em 1 exemplo

In [ ]:
# Implementar função resize(imagem, escala):
# - reduzir e voltar ao tamanho original (ou só reduzir — documentar)
# Implementar função blur(imagem, sigma ou kernel):
# - desfoque gaussiano
# Testar visualmente em 1 exemplo

In [ ]:
# Implementar pipeline combinado:
# crop → resize → JPEG
# (essa é a transformação mais "realista" de compartilhamento)
# Montar um dicionário com TODAS as transformações a testar, ex.:
# {"jpeg_90": fn, "jpeg_70": fn, "crop_25": fn, "blur": fn, "pipeline": fn, ...}

## 7. Avaliação sob transformações
Mesmo SVM do baseline; só o **teste** muda (imagens transformadas).

In [ ]:
# Para cada transformação do dicionário:
#   1) aplicar a transformação em TODAS as imagens do conjunto de TESTE
#   2) para cada feature (FFT, HOG, LBP):
#        - extrair features das imagens transformadas
#        - aplicar o MESMO scaler do treino
#        - prever com o SVM já treinado (NÃO retreinar)
#        - calcular acurácia
#   3) guardar resultado: transformação | feature | acurácia | delta vs baseline
# Dica: começar com subset do teste para validar o loop; depois rodar completo

In [ ]:
# Montar DataFrame/CSV com os resultados
# Colunas sugeridas: transformacao, feature, acuracia, baseline, queda
# Salvar em results/metrics.csv
# Exibir tabela ordenada pela maior queda de desempenho

## 8. Gráficos e análise
Material para poster, slides e relatório.

In [ ]:
# Gráfico de barras: acurácia por transformação, uma cor/série por feature (FFT vs HOG vs LBP)
# Incluir linha ou barra do baseline para comparação
# Salvar figura em results/acuracia_por_transformacao.png

In [ ]:
# (Opcional) heatmap: linhas = transformações, colunas = features, valores = queda de acurácia
# Salvar em results/heatmap_queda.png
# Escrever em markdown/célula de texto: qual feature degradou mais e por quê (ligar aos domínios da disciplina)

## 9. (Opcional) Complemento — outros geradores
Só se der tempo. **Não substitui** o CIFAKE.

In [ ]:
# (Opcional) carregar poucas imagens de ChatGPT/Midjourney/Gemini etc.
# Extrair FFT/HOG/LBP e comparar visualmente os espectros/histogramas
# (Opcional) testar o SVM treinado no CIFAKE nessas imagens
# Anotar no relatório se os padrões mudam entre geradores

## 10. Checklist da entrega (código)

- [ ] Features FFT, HOG e LBP implementadas
- [ ] SVM treinado só em imagens originais (3 modelos)
- [ ] Transformações: JPEG, crop, resize, blur, pipeline
- [ ] Tabela de acurácia baseline vs cada transformação
- [ ] Gráficos salvos em `results/`
- [ ] Modelos `.pkl` salvos em `models/`
- [ ] README com caminho dos modelos e como rodar